In [2]:
# ============================================================
# Physics-Constrained MGB for Colebrook — v4 (Reviewer Revision)
# v4 changes (reviewer revision):
#   - Figure sizes enlarged (FW=18, FH=14; DPI=250) — Reviewer 2
#   - Timing table printed + saved as timing_table.csv — Reviewer 1
#   - Timing table added to output zip
# ============================================================

# 0) Setup
!pip -q install lightgbm==4.5.0

import numpy as np, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd, os, time, warnings, json, zipfile
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
from lightgbm import LGBMRegressor

IN_COLAB = False
try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    pass

os.makedirs("figs", exist_ok=True)

# ============================================================
# FIGURE REGENERATION — v4 REVISION
# Changes: enlarged figures (FW=18,FH=14,Dpi=250), timing table output
# ============================================================
import numpy as np, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.ticker import LogLocator
import pandas as pd, os, time, warnings, json
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
from lightgbm import LGBMRegressor

os.makedirs("figs", exist_ok=True)

# ── helpers ──────────────────────────────────────────────────
def colebrook_white(Re, k_rel, max_iter=60, tol=1e-12):
    Re = np.asarray(Re, float); k_rel = np.asarray(k_rel, float)
    A = -2*np.log10(np.where(k_rel>0,k_rel,1e-12)/3.7 + 12/Re)
    B = -2*np.log10(np.where(k_rel>0,k_rel,1e-12)/3.7 + 2.51*A/Re)
    C = -2*np.log10(np.where(k_rel>0,k_rel,1e-12)/3.7 + 2.51*B/Re)
    x = np.abs(C); ln10 = np.log(10)
    for _ in range(max_iter):
        denom = k_rel/3.7 + 2.51*x/Re
        g  = x + 2*np.log10(np.clip(denom,1e-300,None))
        dg = 1 + 2/ln10*(2.51/Re)/np.clip(denom,1e-300,None)
        step = g/np.clip(dg,1e-16,None); x_new = x - step
        if np.max(np.abs(step)) < tol: x = x_new; break
        x = x_new
    return 1/np.clip(x**2,1e-30,None)

def haaland(Re, k):
    Re=np.asarray(Re,float); k=np.asarray(k,float)
    return (-1.8*np.log10((np.where(k>0,k,1e-12)/3.7)**1.11+6.9/Re))**(-2)

def swamee_jain(Re, k):
    Re=np.asarray(Re,float); k=np.asarray(k,float)
    return 0.25/(np.log10(np.where(k>0,k,1e-12)/3.7+5.74/Re**0.9))**2

def serghides(Re, k):
    Re=np.asarray(Re,float); k=np.asarray(k,float)
    A=-2*np.log10(np.where(k>0,k,1e-12)/3.7+12/Re)
    B=-2*np.log10(np.where(k>0,k,1e-12)/3.7+2.51*A/Re)
    C=-2*np.log10(np.where(k>0,k,1e-12)/3.7+2.51*B/Re)
    return 1/(C*C)

# ── train model ───────────────────────────────────────────────
RE_MIN, RE_MAX = 4e3, 1e8
np.random.seed(42)
re_t = np.logspace(np.log10(RE_MIN), np.log10(RE_MAX), 240)
k_t  = np.linspace(0.0, 0.06, 61)
Re_m, k_m = np.meshgrid(re_t, k_t, indexing='xy')
Re_f = Re_m.ravel(); k_f = k_m.ravel()
f_f  = colebrook_white(Re_f, k_f)
df   = pd.DataFrame({"log10Re":np.log10(Re_f),"k_rel":k_f,"f":f_f})
df["kbin"] = pd.cut(k_f,10,labels=False,include_lowest=True)
tr, rest = train_test_split(df,test_size=0.30,random_state=42,stratify=df["kbin"])
cal, _   = train_test_split(rest,test_size=0.50,random_state=42,stratify=rest["kbin"])
Xtr=tr[["log10Re","k_rel"]].values; ytr=tr["f"].values
Xcal=cal[["log10Re","k_rel"]].values; ycal=cal["f"].values
print("Training model...")
point = LGBMRegressor(n_estimators=1200,learning_rate=0.05,num_leaves=64,min_child_samples=40,
    subsample=0.9,colsample_bytree=0.9,reg_lambda=1.0,objective="l2",
    monotone_constraints=[-1,+1],random_state=42,verbose=-1)
point.fit(Xtr,ytr)
alpha=0.05
ql=LGBMRegressor(n_estimators=800,learning_rate=0.05,num_leaves=64,min_child_samples=40,
    subsample=0.9,colsample_bytree=0.9,reg_lambda=1.0,objective="quantile",alpha=alpha,random_state=42,verbose=-1)
qu=LGBMRegressor(n_estimators=800,learning_rate=0.05,num_leaves=64,min_child_samples=40,
    subsample=0.9,colsample_bytree=0.9,reg_lambda=1.0,objective="quantile",alpha=1-alpha,random_state=42,verbose=-1)
ql.fit(Xtr,ytr); qu.fit(Xtr,ytr)
s=np.maximum(ql.predict(Xcal)-ycal, ycal-qu.predict(Xcal))
q_hat=np.quantile(s,1-alpha)
def CI(X):
    lo=ql.predict(X)-q_hat; up=qu.predict(X)+q_hat
    return np.minimum(lo,up), np.maximum(lo,up)
print("Done.")

# ── shared constants ──────────────────────────────────────────
k_vals = [0.00,0.01,0.02,0.03,0.04,0.05,0.06]
labels_k = [r"$\varepsilon/D=0$",r"$\varepsilon/D=0.01$",r"$\varepsilon/D=0.02$",
            r"$\varepsilon/D=0.03$",r"$\varepsilon/D=0.04$",r"$\varepsilon/D=0.05$",r"$\varepsilon/D=0.06$"]
panels = ["(a)","(b)","(c)","(d)","(e)","(f)","(g)"]
re_line = np.logspace(np.log10(RE_MIN),np.log10(RE_MAX),300)
re_eval = np.logspace(np.log10(RE_MIN),np.log10(RE_MAX),24)
FW, FH = 18, 14   # figure width/height for 2×4 panels
TICK_FS = 8; LBL_FS = 9; PANEL_FS = 9; LEG_FS = 8

def fmt_ax(ax, k, lbl, panel):
    ax.set_xscale('log')
    ax.set_title(f"{panel} Relative Roughness {lbl}", fontsize=PANEL_FS)
    ax.set_xlabel("$Re$", fontsize=LBL_FS)
    ax.tick_params(labelsize=TICK_FS)
    ax.grid(True, which='both', alpha=0.25)

# ── FIG 1: Re vs f ───────────────────────────────────────────
fig, axes = plt.subplots(2,4, figsize=(FW,FH))
axes = axes.ravel()
for i,(k,lbl) in enumerate(zip(k_vals,labels_k)):
    ax = axes[i]
    f_cw = colebrook_white(re_line, np.full_like(re_line,k))
    f_h  = haaland(re_line, np.full_like(re_line,k))
    f_sj = swamee_jain(re_line, np.full_like(re_line,k))
    f_sg = serghides(re_line, np.full_like(re_line,k))
    ax.plot(re_line,f_h, '--',color='tab:orange',lw=1.6,label='Haaland')
    ax.plot(re_line,f_sj,':' ,color='tab:green', lw=1.6,label='Swamee-Jain')
    ax.plot(re_line,f_sg,'-.',color='tab:red',   lw=1.6,label='Serghides')
    ax.plot(re_line,f_cw,'-' ,color='tab:blue',  lw=2.0,label='Colebrook')
    ax.set_ylabel("$f$", fontsize=LBL_FS)
    fmt_ax(ax,k,lbl,panels[i])
    ax.legend(fontsize=LEG_FS, ncol=2)
axes[-1].set_visible(False)
plt.tight_layout()
plt.savefig("figs/fig1.png",dpi=250,bbox_inches='tight'); plt.close()
print("fig1 done")

# ── FIG 2: residuals (Δf) — ALL k including k=0 ─────────────
fig, axes = plt.subplots(2,4, figsize=(FW,FH))
axes = axes.ravel()
for i,(k,lbl) in enumerate(zip(k_vals,labels_k)):
    ax = axes[i]
    f_cw = colebrook_white(re_line,np.full_like(re_line,k))
    ax.plot(re_line,haaland(re_line,np.full_like(re_line,k))-f_cw,
            '--',color='tab:orange',lw=1.6,label='Haaland')
    ax.plot(re_line,swamee_jain(re_line,np.full_like(re_line,k))-f_cw,
            ':' ,color='tab:green', lw=1.6,label='Swamee-Jain')
    ax.plot(re_line,serghides(re_line,np.full_like(re_line,k))-f_cw,
            '-.',color='tab:red',   lw=1.6,label='Serghides')
    ax.axhline(0,color='k',lw=0.8)
    ax.set_ylabel(r"$\Delta f$", fontsize=LBL_FS)
    fmt_ax(ax,k,lbl,panels[i])
    ax.legend(fontsize=LEG_FS)
axes[-1].set_visible(False)
plt.tight_layout()
plt.savefig("figs/fig2.png",dpi=250,bbox_inches='tight'); plt.close()
print("fig2 done")

# ── FIG 3: % error — ALL k including k=0 ────────────────────
fig, axes = plt.subplots(2,4, figsize=(FW,FH))
axes = axes.ravel()
for i,(k,lbl) in enumerate(zip(k_vals,labels_k)):
    ax = axes[i]
    f_cw = colebrook_white(re_line,np.full_like(re_line,k))
    ax.plot(re_line,np.abs(haaland(re_line,np.full_like(re_line,k))-f_cw)/f_cw*100,
            '--',color='tab:orange',lw=1.6,label='Haaland')
    ax.plot(re_line,np.abs(swamee_jain(re_line,np.full_like(re_line,k))-f_cw)/f_cw*100,
            ':' ,color='tab:green', lw=1.6,label='Swamee-Jain')
    ax.plot(re_line,np.abs(serghides(re_line,np.full_like(re_line,k))-f_cw)/f_cw*100,
            '-.',color='tab:red',   lw=1.6,label='Serghides')
    ax.set_ylabel("Error (%)", fontsize=LBL_FS)
    fmt_ax(ax,k,lbl,panels[i])
    ax.legend(fontsize=LEG_FS)
axes[-1].set_visible(False)
plt.tight_layout()
plt.savefig("figs/fig3.png",dpi=250,bbox_inches='tight'); plt.close()
print("fig3 done")

# ── FIG 4: parity plots — scatter for ALL k ─────────────────
fig, axes = plt.subplots(2,4, figsize=(FW,FH))
axes = axes.ravel()
for i,(k,lbl) in enumerate(zip(k_vals,labels_k)):
    ax = axes[i]
    f_cw = colebrook_white(re_eval,np.full_like(re_eval,k))
    f_h  = haaland(re_eval,np.full_like(re_eval,k))
    f_sj = swamee_jain(re_eval,np.full_like(re_eval,k))
    f_sg = serghides(re_eval,np.full_like(re_eval,k))
    ax.scatter(f_cw,f_h, s=25,color='tab:orange',alpha=0.85,label='Haaland',zorder=3)
    ax.scatter(f_cw,f_sj,s=25,color='tab:green', alpha=0.85,label='Swamee-Jain',zorder=3)
    ax.scatter(f_cw,f_sg,s=25,color='tab:red',   alpha=0.85,label='Serghides',zorder=3)
    lim=[min(f_cw.min(),f_h.min(),f_sj.min(),f_sg.min()),
         max(f_cw.max(),f_h.max(),f_sj.max(),f_sg.max())]
    ax.plot(lim,lim,'k-',lw=1.2)
    ax.set_xlabel("Colebrook $f$",fontsize=LBL_FS)
    ax.set_ylabel("Approximate $f$",fontsize=LBL_FS)
    ax.set_title(f"{panels[i]} Relative Roughness {lbl}",fontsize=PANEL_FS)
    ax.tick_params(labelsize=TICK_FS)
    ax.grid(True,alpha=0.25)
    ax.legend(fontsize=LEG_FS)
axes[-1].set_visible(False)
plt.tight_layout()
plt.savefig("figs/fig4.png",dpi=250,bbox_inches='tight'); plt.close()
print("fig4 done")

# ── FIG 5 & 6: bar charts (rough pipe only, k>0 makes sense) ─
k_vals_r = [0.01,0.02,0.03,0.04,0.05,0.06]
panels_r  = ["(a)","(b)","(c)","(d)","(e)","(f)"]
re_b = np.logspace(np.log10(RE_MIN),np.log10(RE_MAX),200)

for fignum,(ylabel,func) in enumerate([
    ("Average % error", lambda f_cw,f_app: np.mean(np.abs(f_app-f_cw)/f_cw)*100),
    ("Std dev % error", lambda f_cw,f_app: np.std(np.abs(f_app-f_cw)/f_cw)*100)
], start=5):
    fig,axes=plt.subplots(2,3,figsize=(FW,FH*0.72))
    axes=axes.ravel()
    for i,k in enumerate(k_vals_r):
        ax=axes[i]
        f_cw=colebrook_white(re_b,np.full_like(re_b,k))
        vals=[func(f_cw,haaland(re_b,np.full_like(re_b,k))),
              func(f_cw,swamee_jain(re_b,np.full_like(re_b,k))),
              func(f_cw,serghides(re_b,np.full_like(re_b,k)))]
        bars=ax.bar(['Haaland','Swamee\n-Jain','Serghides'],vals,
                    color=['tab:orange','tab:green','tab:red'],edgecolor='k',lw=0.8)
        for bar,v in zip(bars,vals):
            ax.text(bar.get_x()+bar.get_width()/2,v+max(vals)*0.02,
                    f'{v:.4f}%',ha='center',va='bottom',fontsize=8)
        ax.set_title(f"{panels_r[i]} "+r"$\varepsilon/D=$"+f"{k}",fontsize=PANEL_FS)
        ax.set_ylabel(ylabel,fontsize=LBL_FS)
        ax.tick_params(labelsize=TICK_FS)
        ax.grid(True,axis='y',alpha=0.25)
    plt.tight_layout()
    plt.savefig(f"figs/fig{fignum}.png",dpi=250,bbox_inches='tight')
    plt.close()
    print(f"fig{fignum} done")

# ── FIG 7: MGB parity (test set) ─────────────────────────────
# need test predictions
_, rest2 = train_test_split(
    pd.DataFrame({"log10Re":np.log10(Re_f),"k_rel":k_f,"f":f_f,
                  "kbin":pd.cut(k_f,10,labels=False,include_lowest=True)}),
    test_size=0.30,random_state=42,
    stratify=pd.cut(k_f,10,labels=False,include_lowest=True))
_, test_df = train_test_split(rest2,test_size=0.50,random_state=42,
    stratify=rest2["kbin"])
X_test = test_df[["log10Re","k_rel"]].values
y_test = test_df["f"].values
y_pred = point.predict(X_test)

fig,ax=plt.subplots(figsize=(7,6))
ax.scatter(y_test,y_pred,s=5,alpha=0.35,color='tab:blue',rasterized=True)
lim=[min(y_test.min(),y_pred.min()),max(y_test.max(),y_pred.max())]
ax.plot(lim,lim,'r-',lw=1.5,label='Perfect fit')
ax.set_xlabel("Colebrook $f$",fontsize=12); ax.set_ylabel("MGB predicted $f$",fontsize=12)
ax.legend(fontsize=10); ax.grid(True,alpha=0.3)
ax.set_title("Parity (Test)",fontsize=11)
plt.tight_layout()
plt.savefig("figs/fig7.png",dpi=250,bbox_inches='tight'); plt.close()
print("fig7 done")

# ── FIG 8: residual vs Re (test set) ─────────────────────────
fig,ax=plt.subplots(figsize=(9,5))
ax.scatter(10**X_test[:,0],y_pred-y_test,s=5,alpha=0.35,color='tab:blue',rasterized=True)
ax.axhline(0,color='k',lw=0.8)
ax.set_xscale('log')
ax.set_xlabel("Reynolds number $Re$",fontsize=12)
ax.set_ylabel("Residual (pred $-$ true)",fontsize=12)
ax.set_title("Residual vs $Re$ (Test)",fontsize=11)
ax.grid(True,which='both',alpha=0.3)
plt.tight_layout()
plt.savefig("figs/fig8.png",dpi=250,bbox_inches='tight'); plt.close()
print("fig8 done")

# ── FIG 9: coverage diagnostic ────────────────────────────────
lo_t,up_t = CI(X_test)
idx = np.argsort(y_pred)
y_s=y_test[idx]; lo_s=lo_t[idx]; up_s=up_t[idx]
cov=(((y_test>=lo_t)&(y_test<=up_t)).mean()*100)
fig,ax=plt.subplots(figsize=(9,5))
ax.plot(range(len(y_s)),y_s-lo_s,color='tab:blue',lw=1,label='$y - L$')
ax.plot(range(len(y_s)),up_s-y_s,color='tab:orange',lw=1,label='$U - y$')
ax.axhline(0,color='k',lw=0.8)
ax.set_xlabel("Samples (sorted by prediction)",fontsize=12)
ax.set_ylabel("Distance to bounds",fontsize=12)
ax.set_title(f"Two-sided Coverage $\\approx$ {cov:.2f}% (nominal 95%)",fontsize=11)
ax.legend(fontsize=10); ax.grid(True,alpha=0.3)
plt.tight_layout()
plt.savefig("figs/fig9.png",dpi=250,bbox_inches='tight'); plt.close()
print("fig9 done")

# ── FIG 10: interval width vs Re ─────────────────────────────
re_ev=np.logspace(np.log10(RE_MIN),np.log10(RE_MAX),24)
k_ev=np.array([0.00,0.01,0.02,0.03,0.04,0.05,0.06])
Re_ev2,k_ev2=np.meshgrid(re_ev,k_ev,indexing='xy')
X_ev=np.column_stack([np.log10(Re_ev2.ravel()),k_ev2.ravel()])
lo_ev,up_ev=CI(X_ev)
w_ev=up_ev-lo_ev
fig,ax=plt.subplots(figsize=(9,5))
ax.scatter(10**X_ev[:,0],w_ev,s=20,alpha=0.7,color='tab:blue')
ax.set_xscale('log')
ax.set_xlabel("Reynolds number $Re$",fontsize=12)
ax.set_ylabel("Prediction interval width",fontsize=12)
ax.set_title(r"Interval Width vs $Re$ on $24\times7$ Evaluation Grid",fontsize=11)
ax.grid(True,which='both',alpha=0.3)
plt.tight_layout()
plt.savefig("figs/fig10.png",dpi=250,bbox_inches='tight'); plt.close()
print("fig10 done")

# ── FIG 11: Nikuradse — legend fixed ─────────────────────────
nik_data={
    0.033:{'Re':np.array([2.37e4,3.13e4,3.84e4,4.94e4,6.42e4,8.00e4,
                          9.72e4,1.18e5,1.52e5,2.37e5,3.00e5,4.00e5]),
           'f': np.array([0.0442,0.0400,0.0375,0.0360,0.0350,0.0342,
                          0.0338,0.0334,0.0330,0.0327,0.0326,0.0325])},
    0.016:{'Re':np.array([2.55e4,3.82e4,5.85e4,8.50e4,1.23e5,1.90e5,
                          3.00e5,4.50e5,7.00e5,1.00e6]),
           'f': np.array([0.0280,0.0254,0.0237,0.0228,0.0222,0.0218,
                          0.0213,0.0211,0.0210,0.0209])},
    0.008:{'Re':np.array([4.00e4,6.50e4,1.00e5,1.60e5,2.60e5,
                          4.20e5,7.00e5,1.20e6,2.00e6]),
           'f': np.array([0.0224,0.0205,0.0193,0.0185,0.0178,
                          0.0173,0.0170,0.0168,0.0167])},
    0.004:{'Re':np.array([6.00e4,1.00e5,1.70e5,3.00e5,
                          5.50e5,1.00e6,2.00e6,4.00e6]),
           'f': np.array([0.0188,0.0175,0.0163,0.0153,
                          0.0145,0.0140,0.0137,0.0135])}
}
colors_n=['#1f77b4','#ff7f0e','#2ca02c','#d62728']
markers_n=['o','s','^','D']
labels_n=[r'$\varepsilon/D=0.033$',r'$\varepsilon/D=0.016$',
          r'$\varepsilon/D=0.008$',r'$\varepsilon/D=0.004$']

fig,ax=plt.subplots(figsize=(10,6.5))
for i,(k_rel,d) in enumerate(nik_data.items()):
    Re_pts=d['Re']; f_pts=d['f']
    Re_l=np.logspace(np.log10(Re_pts.min()*0.75),np.log10(Re_pts.max()*1.25),120)
    X_l=np.column_stack([np.log10(Re_l),np.full_like(Re_l,k_rel)])
    f_mgb=point.predict(X_l); lo_l,up_l=CI(X_l)
    ax.fill_between(Re_l,lo_l,up_l,alpha=0.12,color=colors_n[i])
    ax.plot(Re_l,f_mgb,'-',color=colors_n[i],lw=2.0,
            label=f'MGB {labels_n[i]}')
    ax.scatter(Re_pts,f_pts,marker=markers_n[i],color=colors_n[i],
               s=60,zorder=5,edgecolors='k',lw=0.7,
               label=f'Nikuradse (1933) {labels_n[i]}')

ax.set_xscale('log')
ax.set_xlabel("Reynolds number $Re$",fontsize=13)
ax.set_ylabel("Darcy–Weisbach friction factor $f$",fontsize=13)
ax.set_title("MGB Surrogate vs. Nikuradse (1933) Experimental Data\n"
             "Shaded bands: 95% conformal prediction intervals",fontsize=11)
# Place legend outside plot to avoid overlapping data
ax.grid(True,which='both',alpha=0.3)
# Legend placed BELOW the axes so it does not overlap data
handles, labels_h = ax.get_legend_handles_labels()
ax.legend(handles, labels_h, ncol=2, fontsize=9, framealpha=0.95,
          loc='upper center',
          bbox_to_anchor=(0.5, -0.16),
          borderaxespad=0.0)
plt.tight_layout()
plt.subplots_adjust(bottom=0.28)   # make room for below-axes legend
plt.savefig("figs/fig11.png",dpi=250,bbox_inches='tight'); plt.close()
print("fig11 done")

# ── FIG 12: speed benchmark ───────────────────────────────────
N_SIZES=[1,10,100,1000,10000,100000,1000000]
REPS=7; np.random.seed(1)
t_cw,t_sg,t_mg=[],[],[]
for N in N_SIZES:
    Re_s=np.random.uniform(4e3,1e8,N); k_s=np.random.uniform(0.001,0.05,N)
    X_s=np.column_stack([np.log10(Re_s),k_s])
    ts_cw=[]; ts_sg=[]; ts_mg=[]
    for _ in range(REPS):
        t0=time.perf_counter(); colebrook_white(Re_s,k_s); ts_cw.append(time.perf_counter()-t0)
        t0=time.perf_counter(); serghides(Re_s,k_s);       ts_sg.append(time.perf_counter()-t0)
        t0=time.perf_counter(); point.predict(X_s);         ts_mg.append(time.perf_counter()-t0)
    t_cw.append(np.median(ts_cw)*1e3)
    t_sg.append(np.median(ts_sg)*1e3)
    t_mg.append(np.median(ts_mg)*1e3)
    print(f"  N={N:>8}: CW={t_cw[-1]:.2f}ms  Sg={t_sg[-1]:.2f}ms  MGB={t_mg[-1]:.2f}ms")

# ── Timing table (for manuscript Table 6) ────────────────────
timing_df = pd.DataFrame({
    "N": N_SIZES,
    "Colebrook-White (ms)": [f"{v:.3f}" for v in t_cw],
    "Serghides (ms)":       [f"{v:.3f}" for v in t_sg],
    "MGB surrogate (ms)":   [f"{v:.3f}" for v in t_mg],
})
timing_df.to_csv("timing_table.csv", index=False)
print("\n=== Timing Table (median of 7 runs, Python/NumPy) ===")
print(timing_df.to_string(index=False))
print(f"\nAt N=1,000,000:")
print(f"  Colebrook-White : {t_cw[-1]:.1f} ms")
print(f"  Serghides       : {t_sg[-1]:.1f} ms")
print(f"  MGB surrogate   : {t_mg[-1]:.1f} ms")
print(f"  MGB / C-W ratio : {t_mg[-1]/t_cw[-1]:.1f}x slower")
print(f"  MGB / Serghides : {t_mg[-1]/t_sg[-1]:.1f}x slower")

fig,ax=plt.subplots(figsize=(9,5.5))
ax.loglog(N_SIZES,t_cw,'o-', color='#1f77b4',lw=2,ms=7,label='Colebrook–White (Newton–Raphson)')
ax.loglog(N_SIZES,t_sg,'s--',color='#ff7f0e',lw=2,ms=7,label='Serghides (explicit, 3-step)')
ax.loglog(N_SIZES,t_mg,'^-', color='#2ca02c',lw=2,ms=7,label='MGB surrogate (LightGBM)')
ax.set_xlabel("Number of evaluations $N$",fontsize=12)
ax.set_ylabel("Wall-clock time (ms)",fontsize=12)
ax.set_title("Computational Timing: Colebrook–White vs. Serghides vs. MGB\n"
             "(Python/NumPy; median of 7 runs)",fontsize=10)
ax.legend(fontsize=9,framealpha=0.9)
ax.grid(True,which='both',alpha=0.3)
plt.tight_layout()
plt.savefig("figs/fig12.png",dpi=250,bbox_inches='tight'); plt.close()
print("fig12 done")

# ── FIG 13: density sensitivity ──────────────────────────────
y_ev_true = colebrook_white(Re_ev2.ravel(),k_ev2.ravel())
mask_r = k_ev2.ravel()>0
configs=[(120,31,"120×31\n(3,720)"),(180,46,"180×46\n(8,280)"),
         (240,61,"240×61\n(14,640)\n← this study"),(360,91,"360×91\n(32,760)")]
mapes_s=[]
for n_re_c,n_k_c,lbl in configs:
    re_c=np.logspace(np.log10(RE_MIN),np.log10(RE_MAX),n_re_c)
    k_c=np.linspace(0.0,0.06,n_k_c)
    Re_c2,k_cc=np.meshgrid(re_c,k_c,indexing='xy')
    Re_fc=Re_c2.ravel(); k_fc=k_cc.ravel()
    f_fc=colebrook_white(Re_fc,k_fc)
    df_c=pd.DataFrame({"log10Re":np.log10(Re_fc),"k_rel":k_fc,"f":f_fc})
    df_c["kbin"]=pd.cut(k_fc,10,labels=False,include_lowest=True)
    tr_c,_=train_test_split(df_c,test_size=0.30,random_state=42,stratify=df_c["kbin"])
    mc=LGBMRegressor(n_estimators=1200,learning_rate=0.05,num_leaves=64,min_child_samples=40,
        subsample=0.9,colsample_bytree=0.9,reg_lambda=1.0,objective="l2",
        monotone_constraints=[-1,+1],random_state=42,verbose=-1)
    mc.fit(tr_c[["log10Re","k_rel"]].values,tr_c["f"].values)
    yhat_c=mc.predict(X_ev[mask_r])
    mape_c=np.mean(np.abs((y_ev_true[mask_r]-yhat_c)/y_ev_true[mask_r]))*100
    mapes_s.append(mape_c)
    print(f"  {lbl.replace(chr(10),' ')}: MAPE={mape_c:.4f}%")

fig,ax=plt.subplots(figsize=(9,5.5))
clrs=['#aec7e8','#aec7e8','#1f77b4','#aec7e8']
bars=ax.bar(range(len(configs)),mapes_s,color=clrs,edgecolor='k',lw=0.8)
ax.set_xticks(range(len(configs)))
ax.set_xticklabels([c[2] for c in configs],fontsize=9)
ax.set_ylabel("MAPE on 24×7 grid — rough pipe (%)",fontsize=11)
ax.set_title("Training Grid Density Sensitivity\n(Blue = grid used in this study)",fontsize=11)
ax.axhline(0.012,color='orange',ls=':',lw=2,label='Serghides MAPE (0.012%)')
for bar,m in zip(bars,mapes_s):
    ax.text(bar.get_x()+bar.get_width()/2,m+0.003,f'{m:.4f}%',
            ha='center',va='bottom',fontsize=9)
ax.legend(fontsize=10); ax.set_ylim(0,max(mapes_s)*1.3)
plt.tight_layout()
plt.savefig("figs/fig13_density_sensitivity.png",dpi=250,bbox_inches='tight')
plt.close()
print("fig13 done")

print("\nAll figures done:", sorted(os.listdir("figs")))

# ── Package and download ─────────────────────────────────────
print("\nPreparing download...")
with zipfile.ZipFile("mgb_results_v4.zip", "w") as zf:
    for f in sorted(os.listdir("figs")):
        zf.write(f"figs/{f}", f)
    if os.path.exists("timing_table.csv"):
        zf.write("timing_table.csv", "timing_table.csv")

if IN_COLAB:
    files.download("mgb_results_v4.zip")

print("Done! Download: mgb_results_v4.zip")
print("\nFiles produced:")
for f in sorted(os.listdir("figs")):
    print(f"  {f}")



Training model...
Done.
fig1 done
fig2 done
fig3 done
fig4 done
fig5 done
fig6 done
fig7 done
fig8 done
fig9 done
fig10 done
fig11 done
  N=       1: CW=0.14ms  Sg=0.04ms  MGB=0.68ms
  N=      10: CW=0.16ms  Sg=0.04ms  MGB=0.94ms
  N=     100: CW=0.20ms  Sg=0.05ms  MGB=3.32ms
  N=    1000: CW=0.39ms  Sg=0.11ms  MGB=27.20ms
  N=   10000: CW=1.79ms  Sg=0.66ms  MGB=249.20ms
  N=  100000: CW=17.08ms  Sg=6.72ms  MGB=2702.16ms
  N= 1000000: CW=191.28ms  Sg=77.12ms  MGB=26971.46ms
fig12 done
  120×31 (3,720): MAPE=0.1853%
  180×46 (8,280): MAPE=0.7001%
  240×61 (14,640) ← this study: MAPE=0.1684%
  360×91 (32,760): MAPE=0.1537%
fig13 done

All figures done: ['fig1.png', 'fig10.png', 'fig11.png', 'fig12.png', 'fig13_density_sensitivity.png', 'fig2.png', 'fig3.png', 'fig4.png', 'fig5.png', 'fig6.png', 'fig7.png', 'fig8.png', 'fig9.png']

Preparing download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done! Download: mgb_results_v3.zip

Files produced:
  fig1.png
  fig10.png
  fig11.png
  fig12.png
  fig13_density_sensitivity.png
  fig2.png
  fig3.png
  fig4.png
  fig5.png
  fig6.png
  fig7.png
  fig8.png
  fig9.png
